# Épargne et investissement : même cycle, écart variable · *Saving and investment: same cycle, shifting gap*

Notebook compagnon du chapitre **16. Épargne et investissement : les deux moteurs du financement de l'économie** — [lire l'article](https://nmlab.io/ressources/epargne-et-investissement).
Companion notebook to chapter **16. Saving and Investment: The Two Engines Financing the Economy** — [read the article](https://nmlab.io/en/ressources/saving-and-investment).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# données FRED chargées dans build_figure


from matplotlib.figure import Figure
import matplotlib.pyplot as plt
import pandas as pd

C, W = nm.COLORS, nm.WIDTH_PX


def wrap(ax, x: float, y: float, text: str, *, size: float = 19, color: str | None = None,
         weight: int = 500, ha: str = "left", va: str = "top", width: int = 42,
         lh: float = 1.5) -> int:
    """Écrit un texte replié à ``width`` caractères (coordonnées pixels)."""
    import textwrap
    lines: list[str] = []
    for para in text.split("\n"):
        lines += textwrap.wrap(para, width) or [""]
    ax.text(x, y, "\n".join(lines), fontsize=size, color=color or C["muted"],
            fontweight=weight, ha=ha, va=va, linespacing=lh, zorder=5)
    return len(lines)


def start(height: int = 1010) -> Figure:
    """Figure NMLab au format du site : 1747 px de large, fond sombre."""
    fig = nm.figure(height_px=height)
    fig.patch.set_facecolor(C["bg"])
    return fig


def dec(v: float, lang: str, n: int = 1, sign: bool = False) -> str:
    """Formate un nombre à la française (virgule, moins typographique) ou à l'anglaise."""
    s = f"{v:+.{n}f}" if sign else f"{v:.{n}f}"
    return s.replace("-", "−").replace(".", ",") if lang == "fr" else s


LABELS = {
    "fr": dict(
        title='Épargne et investissement : même cycle, écart variable',
        sub='Épargne brute et investissement intérieur privé brut des États-Unis, en % du PIB',
        l1='Épargne brute',
        l2='Investissement privé brut',
        note="BEA via FRED. L'identité S = I ne vaut qu'une fois comptés l'État et l'extérieur (S = I + X − M) :\nces deux séries partagent le cycle sans se superposer — une identité ne dit jamais qui commande.",
    ),
    "en": dict(
        title='Saving and investment: same cycle, shifting gap',
        sub='US gross saving and gross private domestic investment, % of GDP',
        l1='Gross saving',
        l2='Gross private investment',
        note='BEA via FRED. The S = I identity only holds once government and the rest of the world are counted\n(S = I + X − M): these two series share the cycle without overlapping — identities name no master.',
    ),
}


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab (libellés selon ``lang``)."""
    t = LABELS[lang]
    gdp = nm.load_fred("GDP")
    sav = nm.load_fred("GSAVE") / gdp * 100
    inv = nm.load_fred("GPDI") / gdp * 100
    sav, inv = sav.dropna(), inv.dropna()
    fig = start(1010); ax = nm.axes(fig, left=0.075, bottom=0.185)
    nm.header(fig, t["title"], t["sub"])
    ax.plot(sav.index, sav.values, color=C["blue"], lw=3.1, label=t["l1"], zorder=4)
    ax.plot(inv.index, inv.values, color=C["amber"], lw=3.1, label=t["l2"], zorder=4)
    ax.fill_between(sav.index, sav.values, inv.reindex(sav.index).values,
                    color=C["edge"], alpha=0.45, zorder=2)
    leg = ax.legend(fontsize=19, frameon=True, facecolor=C["bg"], edgecolor=C["edge"],
                    loc="lower left", labelcolor=C["text"])
    leg.get_frame().set_linewidth(1.4)
    ax.grid(color=C["grid"], lw=1.1); ax.set_axisbelow(True)
    ax.tick_params(labelsize=18, colors=C["muted"], length=0)
    ax.yaxis.set_major_formatter(lambda v, p: f"{v:.0f} %")
    for sp in ("top", "right", "bottom", "left"): ax.spines[sp].set_visible(False)
    nm.footer(fig, t["note"])
    return fig


build_figure(LANG)